# Phase 6 — Disfluency Detection (Multi-Label)

**Model:** Wav2Vec2-base fine-tuned for stuttering event detection

**Dataset:** SEP-28k (Apple's stuttering events dataset)

**Key difference from v1:** This is **multi-LABEL** (a clip can have multiple disfluency types). Uses `BCEWithLogitsLoss` + sigmoid, NOT `CrossEntropyLoss` + softmax.

**Setup:** Runtime → Change runtime type → **T4 GPU**

**Time:** Download ~30-60 min, Training ~3-4 hours on T4

**Target:** Macro F1 ≥ 0.45 (research papers report 0.40–0.55 on SEP-28k)

## Cell 1: Install Dependencies & GPU Check

In [ ]:
!pip install -q torch torchaudio transformers accelerate
!pip install -q soundfile librosa pandas scikit-learn

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected! Training will be very slow.")
    print("Go to Runtime -> Change runtime type -> T4 GPU")

## Cell 2: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = "/content/drive/MyDrive/voice_pipeline_models/disfluency"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Models will be saved to: {SAVE_DIR}")

## Cell 3: Clone SEP-28k Repository

In [ ]:
import os

DATA_DIR = "/content/sep28k"
REPO_DIR = os.path.join(DATA_DIR, "repo")
WAVS_DIR = os.path.join(DATA_DIR, "wavs")
CLIPS_DIR = os.path.join(DATA_DIR, "clips")
os.makedirs(WAVS_DIR, exist_ok=True)
os.makedirs(CLIPS_DIR, exist_ok=True)

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/apple/ml-stuttering-events-dataset.git {REPO_DIR}
    print("Repo cloned.")
else:
    print("Repo already exists.")

!ls {REPO_DIR}/
print("\n--- Episodes CSV preview ---")
!head -5 {REPO_DIR}/SEP-28k_episodes.csv
print("\n--- Labels CSV preview ---")
!head -5 {REPO_DIR}/SEP-28k_labels.csv

## Cell 4: Set Kaggle API Token

Paste your API token from Kaggle Settings below.

In [ ]:
!pip install -q kaggle

import os

# Paste your Kaggle API token here
os.environ["KAGGLE_API_TOKEN"] = "KGAT_ac84b0c36a24ad254af5591c12790b33"

print("Kaggle API token set.")

## Cell 5: Download and Extract Kaggle Dataset

In [ ]:
import os

KAGGLE_DIR = "/content/sep28k_kaggle"

# Download from Kaggle
!kaggle datasets download -d vudominhgiang/sep-28k-maintained -p {KAGGLE_DIR}

# Unzip
!unzip -q -o {KAGGLE_DIR}/*.zip -d {KAGGLE_DIR}/

# Explore what we got
print("\n=== Kaggle dataset contents ===")
for root, dirs, fls in os.walk(KAGGLE_DIR):
    level = root.replace(KAGGLE_DIR, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:  # Only show first 2 levels
        for f in sorted(fls)[:10]:
            print(f"{indent}  {f}")
        if len(fls) > 10:
            print(f"{indent}  ... and {len(fls) - 10} more files")

In [ ]:
# Find where the .wav clips actually are in the Kaggle download
import glob

wav_files = glob.glob(f"{KAGGLE_DIR}/**/*.wav", recursive=True)
print(f"Total WAV files found: {len(wav_files)}")

if len(wav_files) > 0:
    # Find the directory containing the most wav files
    from collections import Counter
    dirs = Counter(os.path.dirname(f) for f in wav_files)
    clips_dir_kaggle = dirs.most_common(1)[0][0]
    clip_count = dirs.most_common(1)[0][1]

    print(f"Clips directory: {clips_dir_kaggle}")
    print(f"Clips in that directory: {clip_count}")
    print(f"\nSample filenames:")
    sample = sorted(os.listdir(clips_dir_kaggle))[:5]
    for f in sample:
        print(f"  {f}")

    # Update CLIPS_DIR to point here
    CLIPS_DIR = clips_dir_kaggle
    print(f"\nCLIPS_DIR set to: {CLIPS_DIR}")
else:
    # Maybe they're in a different format (mp3, flac, etc.)
    all_audio = glob.glob(f"{KAGGLE_DIR}/**/*.*", recursive=True)
    extensions = Counter(os.path.splitext(f)[1].lower() for f in all_audio)
    print(f"File extensions found: {extensions}")
    print("No WAV files found. Check the dataset structure above.")

## Cell 6: Validate Audio Files

**CRITICAL:** Corrupt files cause silent training failures — the model trains on zeros/garbage and F1 never improves. This cell checks every clip for corruption, silence, wrong sample rate, and bad duration.

In [ ]:
import soundfile as sf
import numpy as np
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

clip_files = sorted([f for f in os.listdir(CLIPS_DIR) if f.endswith('.wav')])
print(f"Validating {len(clip_files)} audio clips...")
print("This takes 3-5 minutes.\n")

valid_clips = []
corrupt_clips = []
silent_clips = []
wrong_sr_clips = []
too_short_clips = []
too_long_clips = []

for fname in tqdm(clip_files, desc="Validating"):
    fpath = os.path.join(CLIPS_DIR, fname)
    try:
        audio, sr = sf.read(fpath)

        # Check sample rate (16000 expected, but accept others — we resample later)
        if sr not in (16000, 22050, 44100, 48000, 8000):
            wrong_sr_clips.append(fname)
            continue

        if len(audio) == 0:
            corrupt_clips.append(fname)
            continue

        # Mono conversion for length check
        if len(audio.shape) > 1:
            audio_mono = audio.mean(axis=1)
        else:
            audio_mono = audio

        duration = len(audio_mono) / sr
        if duration < 0.5:
            too_short_clips.append(fname)
            continue
        if duration > 5.0:
            too_long_clips.append(fname)
            continue

        rms = np.sqrt(np.mean(audio_mono.astype(np.float32) ** 2))
        if rms < 1e-5:
            silent_clips.append(fname)
            continue

        if np.any(np.isnan(audio_mono)) or np.any(np.isinf(audio_mono)):
            corrupt_clips.append(fname)
            continue

        valid_clips.append(fname)

    except Exception:
        corrupt_clips.append(fname)

print(f"\n{'='*50}")
print(f"VALIDATION RESULTS")
print(f"{'='*50}")
print(f"Valid clips:       {len(valid_clips)}")
print(f"Corrupt/error:     {len(corrupt_clips)}")
print(f"Silent:            {len(silent_clips)}")
print(f"Wrong sample rate: {len(wrong_sr_clips)}")
print(f"Too short (<0.5s): {len(too_short_clips)}")
print(f"Too long (>5s):    {len(too_long_clips)}")
print(f"{'='*50}")
print(f"Usable: {len(valid_clips)} / {len(clip_files)} "
      f"({100*len(valid_clips)/max(len(clip_files),1):.1f}%)")

valid_set = set(valid_clips)

# Remove bad files from disk to save space
removed = 0
for f in corrupt_clips + silent_clips:
    fpath = os.path.join(CLIPS_DIR, f)
    if os.path.exists(fpath):
        os.remove(fpath)
        removed += 1
print(f"\nRemoved {removed} bad files from disk.")

## Cell 7: Parse Labels & Create Clean Dataset

**Key points:**
- SEP-28k labels are annotator counts (0–3), NOT binary. Threshold at ≥2.
- This is **multi-LABEL**: a clip can have Prolongation AND Block simultaneously.
- Filter out clips with NoSpeech, Music, PoorAudioQuality, Unsure ≥ 2.

In [ ]:
import pandas as pd
import numpy as np

labels_csv = f"{REPO_DIR}/SEP-28k_labels.csv"
labels_df = pd.read_csv(labels_csv)

print(f"Total label rows: {len(labels_df)}")
print(f"Columns: {list(labels_df.columns)}")
print(f"\nFirst 3 rows:")
print(labels_df.head(3))

# Detect clip filename format by checking what's on disk
sample_clips = sorted(os.listdir(CLIPS_DIR))[:5]
print(f"\nSample clip filenames on disk: {sample_clips}")
print(f"\nSample label rows:")
print(labels_df[['Show', 'EpId', 'ClipId']].head(5))

# Try multiple naming conventions to find the right one
available_clips = set(os.listdir(CLIPS_DIR))

def try_clip_name(row, fmt_func):
    return fmt_func(row['Show'], row['EpId'], row['ClipId'])

fmt_funcs = [
    ("Show_EpId_ClipId", lambda s, e, c: f"{s}_{e}_{c}.wav"),
    ("Show_EpId_clip+ClipId", lambda s, e, c: f"{s}_{e}_clip{c}.wav"),
]

# Test each format on first 100 rows
best_fmt = None
best_count = 0
for name, func in fmt_funcs:
    count = sum(1 for _, row in labels_df.head(100).iterrows()
                if try_clip_name(row, func) in available_clips)
    print(f"  Format '{name}': {count}/100 matches")
    if count > best_count:
        best_count = count
        best_fmt = func

if best_count == 0:
    print("\nWARNING: No filename format matched! Check clip filenames vs CSV.")
    print("Trying to detect pattern from actual filenames...")
    # Show a few filenames for debugging
    for f in sorted(os.listdir(CLIPS_DIR))[:10]:
        print(f"  {f}")
    raise RuntimeError("Cannot match clip filenames to CSV rows. See above.")

labels_df['clip_filename'] = labels_df.apply(lambda row: best_fmt(row['Show'], row['EpId'], row['ClipId']), axis=1)
print(f"\nUsing format with {best_count}/100 matches")

# Filter to clips we have and that passed validation
labels_df['exists'] = labels_df['clip_filename'].isin(available_clips)
labels_df['valid'] = labels_df['clip_filename'].isin(valid_set)
print(f"\nClips with labels AND on disk: {labels_df['exists'].sum()}")
print(f"Clips with labels AND valid:   {labels_df['valid'].sum()}")

df = labels_df[labels_df['valid']].copy()
print(f"\nWorking dataset size: {len(df)}")

In [ ]:
# Apply quality filters — remove bad clips
THRESHOLD = 2

before = len(df)

# Strip column names in case of whitespace
df.columns = df.columns.str.strip()

if 'NoSpeech' in df.columns:
    df = df[df['NoSpeech'] < THRESHOLD]
if 'Music' in df.columns:
    df = df[df['Music'] < THRESHOLD]
if 'PoorAudioQuality' in df.columns:
    df = df[df['PoorAudioQuality'] < THRESHOLD]
if 'Unsure' in df.columns:
    df = df[df['Unsure'] < THRESHOLD]
# Also try alternate names
if 'DifficultToUnderstand' in df.columns:
    df = df[df['DifficultToUnderstand'] < THRESHOLD]

print(f"Filtered: {before} -> {len(df)} clips ({before - len(df)} removed)")

# Create binary multi-label targets
# These are the 5 disfluency classes
DISFLUENCY_COLS = ['Prolongation', 'Block', 'SoundRep', 'WordRep', 'Interjection']

# Map CSV columns to our standard names (handle different naming conventions)
col_map = {}
for target_col in DISFLUENCY_COLS:
    if target_col in df.columns:
        col_map[target_col] = target_col
    else:
        # Try alternate names
        alternates = {
            'SoundRep': ['SoundRepetition', 'Sound Repetition', 'Sound_Repetition'],
            'WordRep': ['WordRepetition', 'Word Repetition', 'Word_Repetition'],
        }
        found = False
        for alt in alternates.get(target_col, []):
            if alt in df.columns:
                col_map[target_col] = alt
                found = True
                break
        if not found:
            print(f"WARNING: Column '{target_col}' not found in CSV!")
            print(f"  Available columns: {list(df.columns)}")

print(f"\nColumn mapping: {col_map}")

# Create binary labels using threshold >= 2 (majority annotator agreement)
for our_name, csv_col in col_map.items():
    df[f'{our_name}_binary'] = (df[csv_col] >= THRESHOLD).astype(int)

# Fluent column
if 'NoStutteredWords' in df.columns:
    df['Fluent'] = (df['NoStutteredWords'] >= THRESHOLD).astype(int)
else:
    # Default: fluent if no disfluency is active
    df['Fluent'] = 0

# For clips where no disfluency label reaches threshold, mark as fluent
binary_cols = [f'{c}_binary' for c in col_map.keys()]
mask_no_label = df[binary_cols].sum(axis=1) == 0
df.loc[mask_no_label, 'Fluent'] = 1

# Print distribution
print(f"\n{'='*50}")
print(f"CLASS DISTRIBUTION (threshold >= {THRESHOLD})")
print(f"{'='*50}")
print(f"{'Fluent':17s}: {df['Fluent'].sum():>6} ({100*df['Fluent'].mean():.1f}%)")
for col in col_map.keys():
    count = df[f'{col}_binary'].sum()
    pct = 100 * count / len(df)
    print(f"{col:17s}: {count:>6} ({pct:.1f}%)")

# Multi-label stats
n_labels = df[binary_cols].sum(axis=1)
print(f"\nMulti-label stats:")
print(f"  Clips with 0 disfluencies: {(n_labels == 0).sum()}")
print(f"  Clips with 1 disfluency:   {(n_labels == 1).sum()}")
print(f"  Clips with 2+ disfluencies: {(n_labels >= 2).sum()}")

# Add file paths
df['clip_path'] = df['clip_filename'].apply(lambda x: os.path.join(CLIPS_DIR, x))

# Save processed labels
df.to_csv(f"{DATA_DIR}/processed_labels.csv", index=False)
print(f"\nSaved processed labels: {len(df)} samples")

## Cell 8: Speaker-Aware Train/Val/Test Split

Split by **Show** (podcast), NOT randomly. If clips from the same episode appear in train and test, the model memorizes speaker voice, not disfluency patterns.

In [ ]:
import numpy as np

shows = df['Show'].unique()
print(f"Unique shows (podcasts): {len(shows)}")

# 80/10/10 split by show
np.random.seed(42)
show_indices = np.random.permutation(len(shows))
n_train = int(0.8 * len(shows))
n_val = int(0.1 * len(shows))

train_shows = set(shows[show_indices[:n_train]])
val_shows = set(shows[show_indices[n_train:n_train + n_val]])
test_shows = set(shows[show_indices[n_train + n_val:]])

train_df = df[df['Show'].isin(train_shows)].copy()
val_df = df[df['Show'].isin(val_shows)].copy()
test_df = df[df['Show'].isin(test_shows)].copy()

print(f"\nTrain: {len(train_df)} clips from {len(train_shows)} shows")
print(f"Val:   {len(val_df)} clips from {len(val_shows)} shows")
print(f"Test:  {len(test_df)} clips from {len(test_shows)} shows")

# Check balance
for name, split_df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    print(f"\n{name} distribution:")
    for col in col_map.keys():
        count = split_df[f'{col}_binary'].sum()
        print(f"  {col}: {count} ({100*count/len(split_df):.1f}%)")

# Safety check: enough data?
if len(train_df) < 2000:
    print(f"\nWARNING: Only {len(train_df)} training samples.")
    print("Consider using threshold=1 instead of 2, or downloading more episodes.")

## Cell 9: Dataset & DataLoader

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import soundfile as sf
import numpy as np

# Use whatever columns we successfully mapped
ACTIVE_CLASSES = list(col_map.keys())
NUM_CLASSES = len(ACTIVE_CLASSES)
print(f"Active classes ({NUM_CLASSES}): {ACTIVE_CLASSES}")


class SEP28kDataset(Dataset):
    """Multi-label disfluency detection dataset."""

    def __init__(self, dataframe, classes, max_length_sec=3.0, sample_rate=16000, augment=False):
        self.df = dataframe.reset_index(drop=True)
        self.classes = classes
        self.max_length = int(max_length_sec * sample_rate)
        self.sr = sample_rate
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        try:
            audio, sr = sf.read(row['clip_path'])
            if len(audio.shape) > 1:
                audio = audio.mean(axis=1)
            audio = audio.astype(np.float32)

            # Resample if needed
            if sr != self.sr:
                import librosa
                audio = librosa.resample(audio, orig_sr=sr, target_sr=self.sr)
        except Exception:
            audio = np.zeros(self.max_length, dtype=np.float32)

        # Augmentation (training only)
        if self.augment:
            # Speed perturbation
            if np.random.random() < 0.3:
                factor = np.random.uniform(0.9, 1.1)
                indices = np.arange(0, len(audio), factor).astype(int)
                indices = indices[indices < len(audio)]
                audio = audio[indices]

            # Additive noise
            if np.random.random() < 0.3:
                snr_db = np.random.uniform(15, 30)
                signal_power = np.mean(audio ** 2)
                noise_power = signal_power / (10 ** (snr_db / 10))
                noise = np.random.normal(0, np.sqrt(max(noise_power, 1e-10)), len(audio))
                audio = audio + noise.astype(np.float32)

        # Pad or truncate
        if len(audio) > self.max_length:
            start = np.random.randint(0, len(audio) - self.max_length) if self.augment else 0
            audio = audio[start:start + self.max_length]
        elif len(audio) < self.max_length:
            audio = np.pad(audio, (0, self.max_length - len(audio)))

        # Multi-label target
        labels = torch.tensor(
            [row[f'{c}_binary'] for c in self.classes],
            dtype=torch.float32
        )

        return torch.tensor(audio, dtype=torch.float32), labels


# Create datasets
train_dataset = SEP28kDataset(train_df, ACTIVE_CLASSES, augment=True)
val_dataset = SEP28kDataset(val_df, ACTIVE_CLASSES, augment=False)
test_dataset = SEP28kDataset(test_df, ACTIVE_CLASSES, augment=False)

BATCH_SIZE = 16

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=2, pin_memory=True)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

# Verify a batch
batch_audio, batch_labels = next(iter(train_loader))
print(f"\nBatch shapes:")
print(f"  Audio:  {batch_audio.shape}")   # (16, 48000)
print(f"  Labels: {batch_labels.shape}")   # (16, 5)
print(f"  Label example: {batch_labels[0]}")

## Cell 10: Model Architecture

Wav2Vec2-base with **multi-label** classification head.

Key: outputs raw logits (no softmax/sigmoid) — `BCEWithLogitsLoss` applies sigmoid internally.

In [ ]:
import torch
import torch.nn as nn
from transformers import Wav2Vec2Model


class DisfluencyDetector(nn.Module):
    """Wav2Vec2-base with multi-label disfluency classification head."""

    def __init__(self, num_classes, model_name="facebook/wav2vec2-base", freeze_layers=6, dropout=0.3):
        super().__init__()
        self.wav2vec2 = Wav2Vec2Model.from_pretrained(model_name)
        hidden_size = self.wav2vec2.config.hidden_size  # 768

        # Freeze feature extractor completely
        self.wav2vec2.feature_extractor._freeze_parameters()

        # Freeze first N transformer layers
        for i, layer in enumerate(self.wav2vec2.encoder.layers):
            if i < freeze_layers:
                for param in layer.parameters():
                    param.requires_grad = False

        # Classification head — NO softmax here
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

        total = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"Model: {model_name}")
        print(f"  Total params:     {total:>12,}")
        print(f"  Trainable params: {trainable:>12,} ({100*trainable/total:.1f}%)")
        print(f"  Frozen layers:    first {freeze_layers} of 12")

    def forward(self, input_values):
        outputs = self.wav2vec2(input_values)
        hidden = outputs.last_hidden_state          # (batch, time, 768)
        pooled = hidden.mean(dim=1)                 # (batch, 768)
        logits = self.classifier(pooled)            # (batch, num_classes)
        return logits


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DisfluencyDetector(num_classes=NUM_CLASSES, freeze_layers=6).to(device)
print(f"\nDevice: {device}")

## Cell 11: Focal Loss for Multi-Label

Standard BCE doesn't work with 5% positive rates. Focal loss down-weights easy negatives so the model focuses on hard cases. `gamma=2` means easy examples contribute ~25x less to the loss.

In [ ]:
class FocalBCELoss(nn.Module):
    """Focal loss for multi-label binary classification."""

    def __init__(self, gamma=2.0, pos_weight=None):
        super().__init__()
        self.gamma = gamma
        self.pos_weight = pos_weight

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        bce = nn.functional.binary_cross_entropy_with_logits(
            logits, targets, reduction='none',
            pos_weight=self.pos_weight
        )
        p_t = probs * targets + (1 - probs) * (1 - targets)
        focal_weight = (1 - p_t) ** self.gamma
        loss = focal_weight * bce
        return loss.mean()


# Compute positive weights from training data
train_labels_tensor = torch.stack([
    torch.tensor([row[f'{c}_binary'] for c in ACTIVE_CLASSES], dtype=torch.float32)
    for _, row in train_df.iterrows()
])

pos_counts = train_labels_tensor.sum(dim=0)
neg_counts = len(train_df) - pos_counts
pos_weight = (neg_counts / pos_counts.clamp(min=1)).to(device)

print("Class positive weights (higher = rarer, gets more attention):")
for i, col in enumerate(ACTIVE_CLASSES):
    print(f"  {col:17s}: {pos_weight[i]:.1f} "
          f"({int(pos_counts[i])} pos / {len(train_df)} total)")

criterion = FocalBCELoss(gamma=2.0, pos_weight=pos_weight)
print(f"\nUsing Focal BCE Loss with gamma=2.0")

## Cell 12: Optimizer & Scheduler

**Critical:** Encoder LR must be very small (1e-5) or pretrained representations get destroyed. Classifier head uses 5e-4.

In [ ]:
from transformers import get_cosine_schedule_with_warmup

# Differential learning rates
optimizer = torch.optim.AdamW([
    {"params": [p for n, p in model.wav2vec2.named_parameters() if p.requires_grad],
     "lr": 1e-5, "weight_decay": 0.01},
    {"params": model.classifier.parameters(),
     "lr": 5e-4, "weight_decay": 0.01},
])

EPOCHS = 20
total_steps = len(train_loader) * EPOCHS
warmup_steps = int(0.1 * total_steps)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

scaler = torch.amp.GradScaler()

print(f"Optimizer: AdamW")
print(f"  Encoder LR:    1e-5")
print(f"  Classifier LR: 5e-4")
print(f"Scheduler: Cosine with {warmup_steps} warmup steps")
print(f"Epochs: {EPOCHS}")
print(f"Total steps: {total_steps}")
print(f"FP16: Enabled")

## Cell 13: Evaluation Function

In [ ]:
from sklearn.metrics import f1_score
import numpy as np


def evaluate(model, dataloader, criterion, device, classes, threshold=0.5):
    """Evaluate with per-class and macro F1 for multi-label."""
    model.eval()
    all_preds = []
    all_labels = []
    total_loss = 0
    n_batches = 0

    with torch.no_grad():
        for audio, labels in dataloader:
            audio = audio.to(device)
            labels = labels.to(device)

            with torch.amp.autocast(device_type='cuda'):
                logits = model(audio)
                loss = criterion(logits, labels)

            total_loss += loss.item()
            n_batches += 1

            probs = torch.sigmoid(logits)
            preds = (probs >= threshold).float()

            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    all_preds = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)

    per_class_f1 = {}
    for i, col in enumerate(classes):
        f1 = f1_score(all_labels[:, i], all_preds[:, i], zero_division=0)
        per_class_f1[col] = f1

    macro_f1 = np.mean(list(per_class_f1.values()))
    exact_match = np.all(all_preds == all_labels, axis=1).mean()

    return {
        "loss": total_loss / max(n_batches, 1),
        "macro_f1": macro_f1,
        "per_class_f1": per_class_f1,
        "exact_match": exact_match,
    }


print("Evaluation function ready.")

## Cell 14: Training Loop

Trains with FP16 mixed precision, gradient clipping, early stopping (patience=5).

In [ ]:
import time

best_val_f1 = 0.0
patience_counter = 0
PATIENCE = 5
history = []

print(f"{'='*70}")
print(f"TRAINING STARTED - {EPOCHS} epochs, {len(train_loader)} batches/epoch")
print(f"{'='*70}\n")

for epoch in range(EPOCHS):
    model.train()
    train_losses = []
    epoch_start = time.time()

    for step, (audio, labels) in enumerate(train_loader):
        audio = audio.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        with torch.amp.autocast(device_type='cuda'):
            logits = model(audio)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        train_losses.append(loss.item())

        if (step + 1) % 50 == 0:
            print(f"  Epoch {epoch+1} Step {step+1}/{len(train_loader)} "
                  f"Loss: {np.mean(train_losses[-50:]):.4f}")

    # Validate
    val_metrics = evaluate(model, val_loader, criterion, device, ACTIVE_CLASSES)
    epoch_time = time.time() - epoch_start

    log_entry = {
        "epoch": epoch + 1,
        "train_loss": float(np.mean(train_losses)),
        "val_loss": val_metrics["loss"],
        "val_macro_f1": val_metrics["macro_f1"],
        "val_per_class_f1": val_metrics["per_class_f1"],
        "epoch_time_sec": epoch_time,
    }
    history.append(log_entry)

    print(f"\nEpoch {epoch+1}/{EPOCHS} ({epoch_time:.0f}s)")
    print(f"  Train Loss:   {np.mean(train_losses):.4f}")
    print(f"  Val Loss:     {val_metrics['loss']:.4f}")
    print(f"  Val Macro F1: {val_metrics['macro_f1']:.4f}")
    print(f"  Per-class F1:")
    for cls, f1 in val_metrics['per_class_f1'].items():
        print(f"    {cls:17s}: {f1:.4f}")

    # Save best model
    if val_metrics['macro_f1'] > best_val_f1:
        best_val_f1 = val_metrics['macro_f1']
        patience_counter = 0

        torch.save({
            'model_state_dict': model.state_dict(),
            'epoch': epoch + 1,
            'val_macro_f1': best_val_f1,
            'per_class_f1': val_metrics['per_class_f1'],
            'classes': ACTIVE_CLASSES,
            'num_classes': NUM_CLASSES,
            'history': history,
        }, f"{SAVE_DIR}/best_model.pt")

        print(f"  >>> NEW BEST MODEL SAVED (Macro F1 = {best_val_f1:.4f})")
    else:
        patience_counter += 1
        print(f"  No improvement ({patience_counter}/{PATIENCE})")

        if patience_counter >= PATIENCE:
            print(f"\n{'='*50}")
            print(f"EARLY STOPPING at epoch {epoch+1}")
            print(f"{'='*50}")
            break

    print()

print(f"\nTraining complete. Best Val Macro F1: {best_val_f1:.4f}")

## Cell 15: Test Set Evaluation

In [ ]:
# Load best model
checkpoint = torch.load(f"{SAVE_DIR}/best_model.pt", map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded best model from epoch {checkpoint['epoch']}")

# Evaluate on test set
test_metrics = evaluate(model, test_loader, criterion, device, ACTIVE_CLASSES)

print(f"\n{'='*60}")
print(f"FINAL TEST SET RESULTS")
print(f"{'='*60}")
print(f"Test Loss:      {test_metrics['loss']:.4f}")
print(f"Test Macro F1:  {test_metrics['macro_f1']:.4f}")
print(f"Exact Match:    {test_metrics['exact_match']:.4f}")
print(f"\nPer-class F1:")

# Research-realistic targets
targets = {
    'Prolongation': 0.45, 'Block': 0.25, 'SoundRep': 0.40,
    'WordRep': 0.35, 'Interjection': 0.60
}

for cls, f1 in test_metrics['per_class_f1'].items():
    target = targets.get(cls, 0.40)
    status = "PASS" if f1 >= target else "BELOW"
    print(f"  [{status:5s}] {cls:17s}: {f1:.4f}  (target: {target:.2f})")

target_met = test_metrics['macro_f1'] >= 0.45
print(f"\n{'='*60}")
print(f"Target Macro F1 >= 0.45: {'PASSED' if target_met else 'BELOW TARGET'}")
print(f"{'='*60}")

# Research context
print(f"\nContext: Apple (2021) got Macro F1=0.43, Wav2Vec2 papers got ~0.54")
print(f"Getting 0.45+ means your model is learning real disfluency patterns.")

if not target_met:
    print("\nSuggestions to improve:")
    print("  1. Unfreeze more layers (freeze_layers=3 instead of 6)")
    print("  2. Increase epochs to 30 with patience=7")
    print("  3. Try encoder LR 5e-6")
    print("  4. Try threshold=1 for more training data")
    print("  5. Try binary detection first (fluent vs any-disfluency)")

## Cell 16: Save Everything to Google Drive

In [ ]:
import json

# Save test metrics
with open(f"{SAVE_DIR}/test_metrics.json", "w") as f:
    json.dump({
        "macro_f1": test_metrics["macro_f1"],
        "exact_match": test_metrics["exact_match"],
        "per_class_f1": test_metrics["per_class_f1"],
        "classes": ACTIVE_CLASSES,
        "num_classes": NUM_CLASSES,
        "task": "multi-label",
    }, f, indent=2)

# Save training history
with open(f"{SAVE_DIR}/history.json", "w") as f:
    json.dump(history, f, indent=2, default=str)

print(f"Saved to {SAVE_DIR}/:")
!ls -lh {SAVE_DIR}/

print(f"\n{'='*60}")
print(f"DONE! Download best_model.pt from Google Drive to your Mac:")
print(f"  Place at: ~/Desktop/Claude-assistant/models/disfluency/best_model.pt")
print(f"{'='*60}")

---

## Troubleshooting

### If podcast downloads fail completely (< 5000 clips)

Run the cell below to try the Kaggle maintained version instead:

In [ ]:
# === ALTERNATIVE: Kaggle maintained dataset ===
# Only run this if Cell 4/5 didn't produce enough clips.
#
# Step 1: Upload your kaggle.json API key:
#   from google.colab import files
#   files.upload()  # upload kaggle.json
#   !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
#
# Step 2: Download
#   !pip install -q kaggle
#   !kaggle datasets download -d vudominhgiang/sep-28k-maintained
#   !unzip -q sep-28k-maintained.zip -d /content/sep28k_kaggle/
#   CLIPS_DIR = "/content/sep28k_kaggle"  # Update path, then re-run from Cell 6
#
print("Uncomment the lines above if you need the Kaggle alternative.")